# DiffuGPT-S: Final-Token Prediction Probing across Layers and Diffusion Time

This notebook is the **complement** to the part-of-speech / token-class probing plot. That experiment showed that **shallow** layers (e.g. Layer 0) are best at predicting a masked token's *linguistic class* (structural information), and that this ability degrades with depth.

Here we ask the opposite question: **which layers can predict the exact final token of a still-masked position?** The hypothesis is that *token prediction is a deep-layer job*, so we expect the curves to invert relative to the POS plot:

- **Shallow layers (Layer 0): low** final-token accuracy.
- **Deep layers (Layer 10): high** final-token accuracy.

We measure this two ways:

1. **Logit lens (headline, no training):** project each layer's residual stream at a masked position through the model's own final norm + unembedding, and check whether the top-1 token equals the final unmasked token.
2. **Trained linear probe (optional confirmation):** train a per-layer linear classifier from the hidden state to the final token id (restricted to observed tokens), then evaluate masked-token accuracy.

Both are plotted as **masked-token accuracy vs. diffusion timestep**, stratified by layer, to mirror the POS plot exactly. Runtime target: Colab/Jupyter with a **T4** GPU.

## 1. Install dependencies and fetch DiffuGPT helper files

In [ ]:
!pip -q install "torch" "transformers==4.44.2" "huggingface_hub" "safetensors" "pandas" "matplotlib" "tqdm" "accelerate"
!curl -L -o model.py https://raw.githubusercontent.com/HKUNLP/DiffuLLaMA/main/model.py
!curl -L -o attention_patch.py https://raw.githubusercontent.com/HKUNLP/DiffuLLaMA/main/attention_patch.py

## 2. Imports and configuration

`LAYERS_TO_PROBE` defaults to `[0, 5, 10]` to match the POS plot (shallow / middle / deep). DiffuGPT-S is a 12-layer GPT-2-small-style model, so Layer 10 is near the top. Set `RUN_TRAINED_PROBE = True` to additionally train linear probes (slower).

In [ ]:
import json
import math
import random
import string
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
plt.rcParams["pdf.fonttype"] = 42  # Type 1 / embedded fonts in PDF outputs
plt.rcParams["ps.fonttype"] = 42
import numpy as np
import pandas as pd
import torch
import torch.distributions as dists
import torch.nn as nn
from tqdm.auto import tqdm
from transformers import AutoConfig, AutoTokenizer

from model import DiscreteDiffusionModel, get_anneal_attn_mask, top_p_logits

MODEL_NAME = "diffusionfamily/diffugpt-s"
BASE_MODEL_NAME = "gpt2"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

DIFFUSION_STEPS = 64
GEN_LEN = 96
LOGITS_TEMP = 0.95
TOPP_TEMP = 0.9
SHIFT = True
SEED = 42

NUM_PROMPTS_PER_TASK = 50          # 100 sequences total (50 reasoning + 50 creative)
LAYERS_TO_PROBE = [0, 5, 10]       # shallow / middle / deep, to match the POS plot

RUN_TRAINED_PROBE = False          # set True to also train per-layer linear probes
PROBE_TRAIN_FRAC = 0.7             # fraction of sequences used to train the probes
PROBE_EPOCHS = 8
PROBE_BATCH = 512
PROBE_LR = 1e-3
MAX_PROBE_SAMPLES = 60000          # cap collected samples to stay within T4 RAM

OUT_DIR = Path("diffugpt_s_layer_token_probe_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Colors chosen to echo the POS plot (shallow=yellow-green, middle=pink, deep=blue).
LAYER_COLORS = {0: "#1f77b4", 5: "#ff7f0e", 10: "#2ca02c", 11: "#9467bd"}
def layer_color(L, i): return LAYER_COLORS.get(L, ["#1f77b4", "#ff7f0e", "#2ca02c", "#9467bd", "#d62728"][i % 5])

print("device:", DEVICE)
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

## 3. Load DiffuGPT-S

In [ ]:
torch.manual_seed(SEED)
random.seed(SEED)

config = AutoConfig.from_pretrained(MODEL_NAME)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.mask_token_id is None:
    raise ValueError("DiffuGPT tokenizer is expected to define mask_token_id.")
MASK_ID = int(tokenizer.mask_token_id)

model = DiscreteDiffusionModel.from_pretrained(
    MODEL_NAME, model=BASE_MODEL_NAME, config=config, tokenizer=tokenizer, device=DEVICE,
).to(DEVICE)
model.eval()
if DEVICE == "cuda":
    model = model.half()

NUM_LAYERS = len(model.denoise_model.h)
FINAL_NORM = getattr(model.denoise_model, "ln_f", None)
LM_HEAD = model.lm_head
print("mask_token_id:", MASK_ID, "| num transformer layers:", NUM_LAYERS, "| vocab:", model.vocab_size)
assert max(LAYERS_TO_PROBE) < NUM_LAYERS, "LAYERS_TO_PROBE has a layer index >= num layers"

## 4. Stratified prompt bank (reasoning vs creative)

In [ ]:
rng = random.Random(SEED)
REASONING_TEMPLATES = [
    "Question: A shop sold {a} items on Monday and half as many on Tuesday. How many in total? Answer step by step.\nAnswer:",
    "Question: A train travels {a} miles in {b} hours. What is its average speed in miles per hour? Explain briefly.\nAnswer:",
    "Question: A box has {a} red marbles and {b} blue marbles. What fraction are red? Explain step by step.\nAnswer:",
    "Question: What is {a} plus {b}? Show your reasoning step by step.\nAnswer:",
    "Question: If a book has {a} pages and you read {b} per day, how many days to finish? Explain.\nAnswer:",
]
CREATIVE_TEMPLATES = [
    "Write a short, vivid paragraph about a city waking up after rain:\n",
    "Continue this story in a whimsical style: The old library only opened its hidden door when\n",
    "Describe a quiet morning in a {place} using vivid sensory detail:\n",
    "Write the opening of a story about a {object} that could remember the future:\n",
    "Tell a gentle tale about a {object} who wanted to see the sea:\n",
]
PLACES = ["a harbor town", "a mountain village", "an old forest", "a desert outpost", "a riverside market"]
OBJECTS = ["lantern", "music box", "compass", "paper boat", "clockwork bird"]

def build_prompts(num_per_task):
    prompts = []
    for i in range(num_per_task):
        prompts.append({"id": f"reasoning_{i}", "task": "reasoning",
                        "prompt": REASONING_TEMPLATES[i % len(REASONING_TEMPLATES)].format(a=rng.randint(12, 96), b=rng.randint(2, 12))})
    for i in range(num_per_task):
        prompts.append({"id": f"creative_{i}", "task": "creative",
                        "prompt": CREATIVE_TEMPLATES[i % len(CREATIVE_TEMPLATES)].format(place=rng.choice(PLACES), object=rng.choice(OBJECTS))})
    return prompts

PROMPTS = build_prompts(NUM_PROMPTS_PER_TASK)
print(f"{len(PROMPTS)} prompts ({NUM_PROMPTS_PER_TASK} per task)")

## 5. Diffusion generation with histories

Same proven DiffuGPT-S sampler used in the issue 44/45 notebook. It stores `xt_history` (the visible/masked sequence at each step) and `final_ids` (ground truth). We re-forward each saved step later with `output_hidden_states=True` to read every layer's residual stream in a single pass.

In [ ]:
@dataclass
class HistoryResult:
    prompt_id: str
    task: str
    prompt: str
    prefix_len: int
    final_ids: list
    xt_history: list

def make_prefix_inputs(prompt, gen_len):
    prefix = [tokenizer.bos_token_id] + tokenizer.encode(prompt, add_special_tokens=False)
    if len(prefix) >= gen_len:
        prefix = prefix[: gen_len - 1]
    src_mask = [1] * len(prefix) + [0] * (gen_len - len(prefix))
    x0 = prefix + [0] * (gen_len - len(prefix))
    return {"input_ids": torch.tensor([x0], dtype=torch.long),
            "src_mask": torch.tensor([src_mask], dtype=torch.long), "prefix_len": len(prefix)}

@torch.inference_mode()
def generate_with_history(prompt_record, seed=SEED):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    inputs = make_prefix_inputs(prompt_record["prompt"], GEN_LEN)
    x = inputs["input_ids"].to(DEVICE)
    src_mask = inputs["src_mask"].bool().to(DEVICE)
    prefix_len = int(inputs["prefix_len"])

    x_embed = model.get_embeds(x)
    attention_mask = get_anneal_attn_mask(x.size(1), x.size(0), dtype=x_embed.dtype, device=x.device, attn_mask_ratio=1.0)
    maskable_mask = ~src_mask
    xt = x.masked_fill(maskable_mask, MASK_ID)
    xt_history = []

    def propose_x0(logits):
        filt = top_p_logits(logits / LOGITS_TEMP, p=TOPP_TEMP)
        scores = torch.log_softmax(filt, dim=-1)
        x0 = dists.Categorical(logits=scores).sample()
        if SHIFT:
            x0 = torch.cat([x[:, 0:1], x0[:, :-1]], dim=1)
        return xt.masked_scatter(maskable_mask, x0[maskable_mask])

    logits = model(xt, attention_mask=attention_mask)
    xt_history.append(xt.detach().cpu()[0].tolist())
    x0 = propose_x0(logits)

    for t in range(DIFFUSION_STEPS - 1, 0, -1):
        p_to_x0 = 1 / (t + 1)
        masked_to_x0 = maskable_mask & (torch.rand_like(x0, dtype=torch.float) < p_to_x0)
        xt.masked_scatter_(masked_to_x0, x0[masked_to_x0])
        maskable_mask = maskable_mask.masked_fill(masked_to_x0, False)
        logits = model(xt, attention_mask=attention_mask)
        xt_history.append(xt.detach().cpu()[0].tolist())
        x0 = propose_x0(logits)

    final_ids = x0.detach().cpu()[0].tolist()
    return HistoryResult(prompt_record["id"], prompt_record["task"], prompt_record["prompt"],
                         prefix_len, final_ids, xt_history)

histories = []
for i, rec in enumerate(tqdm(PROMPTS, desc="generate histories")):
    histories.append(generate_with_history(rec, seed=SEED + i))
print("generated", len(histories), "histories")

## 6. Hidden-state helper and logit lens

`model_hidden_states_for_ids` runs one forward pass with `output_hidden_states=True`. `hidden_states[L+1]` is the residual stream *after* transformer layer `L` (index 0 is the embedding). The logit lens applies the model's final norm + unembedding to a residual vector.

Because DiffuGPT uses `SHIFT=True`, the token at position `p` is predicted from the residual at position `p-1`, so we always probe the **source position** `p-1`.

In [ ]:
@torch.inference_mode()
def model_hidden_states_for_ids(ids):
    input_ids = torch.tensor(ids, dtype=torch.long, device=DEVICE).unsqueeze(0)
    x_embed = model.get_embeds(input_ids)
    attention_mask = get_anneal_attn_mask(input_ids.size(1), input_ids.size(0),
                                          dtype=x_embed.dtype, device=input_ids.device, attn_mask_ratio=1.0)
    outputs = model.denoise_model(inputs_embeds=x_embed, attention_mask=attention_mask,
                                  output_hidden_states=True, return_dict=True)
    return outputs.hidden_states  # tuple len = NUM_LAYERS + 1

@torch.inference_mode()
def logit_lens_argmax(vecs, apply_final_norm=True):
    # vecs: [M, hidden] -> argmax token id per row
    if apply_final_norm and FINAL_NORM is not None:
        vecs = FINAL_NORM(vecs)
    logits = LM_HEAD(vecs)
    return torch.argmax(logits, dim=-1)

## 7. Logit-lens probe: masked-token final-prediction accuracy per layer per step

For each sequence and diffusion step, we take the positions that are **still masked**, read each probed layer's residual at the source position `p-1`, decode it with the logit lens, and check whether the top-1 token equals the eventual final token at `p`.

In [ ]:
def generated_positions(hist):
    return list(range(hist.prefix_len, GEN_LEN))

ll_rows = []
for hist in tqdm(histories, desc="logit-lens probe"):
    gpos = generated_positions(hist)
    final = hist.final_ids
    n_steps = len(hist.xt_history)
    for t in range(n_steps):
        xt = hist.xt_history[t]
        masked = [p for p in gpos if int(xt[p]) == MASK_ID]
        if not masked:
            for L in LAYERS_TO_PROBE:
                ll_rows.append({"prompt_id": hist.prompt_id, "task": hist.task, "step": t,
                                "layer": L, "n_masked": 0, "n_correct": 0})
            continue
        src_idx = torch.tensor([p - 1 for p in masked], device=DEVICE)
        tgt = torch.tensor([int(final[p]) for p in masked], device=DEVICE)
        hidden = model_hidden_states_for_ids(xt)
        for L in LAYERS_TO_PROBE:
            vecs = hidden[L + 1][0, src_idx]            # residual after layer L at source positions
            pred = logit_lens_argmax(vecs)
            n_correct = int((pred == tgt).sum().item())
            ll_rows.append({"prompt_id": hist.prompt_id, "task": hist.task, "step": t,
                            "layer": L, "n_masked": len(masked), "n_correct": n_correct})

ll_df = pd.DataFrame(ll_rows)
ll_df.to_csv(OUT_DIR / "logit_lens_masked_token_accuracy_raw.csv", index=False)
# Aggregate to accuracy per (layer, step): pooled over sequences (sum correct / sum masked).
ll_curve = (ll_df.groupby(["layer", "step"], as_index=False)
            .agg(n_masked=("n_masked", "sum"), n_correct=("n_correct", "sum")))
ll_curve["accuracy"] = ll_curve["n_correct"] / ll_curve["n_masked"].clip(lower=1)
ll_curve.to_csv(OUT_DIR / "logit_lens_masked_token_accuracy_curve.csv", index=False)
print(ll_curve.groupby("layer")["accuracy"].mean())

## 8. Headline plot: Final-Token Prediction Probing Accuracy over Diffusion Timesteps

This is the direct counterpart to the POS plot. We expect the ordering to be **inverted**: deep layers on top, shallow layers at the bottom.

In [ ]:
with plt.style.context("default"):
    plt.figure(figsize=(9.2, 5.6))
    for i, L in enumerate(LAYERS_TO_PROBE):
        sub = ll_curve[ll_curve["layer"] == L].sort_values("step")
        tag = "Shallow" if L == min(LAYERS_TO_PROBE) else ("Deep" if L == max(LAYERS_TO_PROBE) else "Middle")
        plt.plot(sub["step"], sub["accuracy"], color=layer_color(L, i), linewidth=1.8,
                 label=f"Layer {L} ({tag})")
    plt.xlabel("Diffusion Timestep (t)")
    plt.ylabel("Masked Token Accuracy")
    plt.title("Final Token Prediction Probing Accuracy over Diffusion Timesteps (logit lens)")
    plt.ylim(0, 1.0)
    plt.grid(alpha=0.25, linestyle="--")
    plt.legend(loc="upper right")
    plt.tight_layout()
    plt.rcParams["pdf.fonttype"] = 42  # Type 1 fonts before saving
    plt.savefig(OUT_DIR / "final_token_prediction.pdf", format="pdf", bbox_inches="tight")
    plt.show()

## 9. (Optional) Trained linear probe

To mirror the POS plot's *trained-probe* methodology exactly, this section trains a per-layer linear classifier `hidden -> final token id`, restricted to the set of final tokens observed in the training sequences. We split by sequence (no leakage), train on masked-position samples, and evaluate masked-token accuracy per diffusion step on held-out sequences.

Set `RUN_TRAINED_PROBE = True` in the config cell to run it.

In [ ]:
trained_curve = None
if RUN_TRAINED_PROBE:
    # ---- 1) collect samples: hidden vecs at source positions for masked tokens ----
    feats = {L: [] for L in LAYERS_TO_PROBE}
    labels, steps, seqids = [], [], []
    collected = 0
    for si, hist in enumerate(tqdm(histories, desc="probe: collect")):
        if collected >= MAX_PROBE_SAMPLES:
            break
        gpos = generated_positions(hist)
        final = hist.final_ids
        for t in range(len(hist.xt_history)):
            xt = hist.xt_history[t]
            masked = [p for p in gpos if int(xt[p]) == MASK_ID]
            if not masked:
                continue
            src_idx = torch.tensor([p - 1 for p in masked], device=DEVICE)
            hidden = model_hidden_states_for_ids(xt)
            for L in LAYERS_TO_PROBE:
                feats[L].append(hidden[L + 1][0, src_idx].float().cpu())
            labels.extend(int(final[p]) for p in masked)
            steps.extend([t] * len(masked))
            seqids.extend([si] * len(masked))
            collected += len(masked)
            if collected >= MAX_PROBE_SAMPLES:
                break
    for L in LAYERS_TO_PROBE:
        feats[L] = torch.cat(feats[L], dim=0)
    labels = np.array(labels); steps = np.array(steps); seqids = np.array(seqids)
    print("collected samples:", len(labels))

    # ---- 2) split by sequence ----
    uniq_seq = sorted(set(seqids.tolist()))
    rng2 = random.Random(SEED); rng2.shuffle(uniq_seq)
    n_train = max(1, int(round(PROBE_TRAIN_FRAC * len(uniq_seq))))
    train_seqs = set(uniq_seq[:n_train]); 
    is_train = np.array([s in train_seqs for s in seqids])

    # label map from TRAIN tokens only
    train_tokens = sorted(set(labels[is_train].tolist()))
    tok2cls = {tid: i for i, tid in enumerate(train_tokens)}
    C = len(train_tokens)
    print(f"train sequences: {n_train}/{len(uniq_seq)} | classes (observed tokens): {C}")

    def to_cls(arr): return np.array([tok2cls.get(int(t), -1) for t in arr])
    y_all = to_cls(labels)

    # ---- 3) train one linear probe per layer ----
    trained_rows = []
    for L in LAYERS_TO_PROBE:
        X = feats[L]
        Xtr = X[is_train]; ytr = torch.tensor(y_all[is_train], dtype=torch.long)
        keep = ytr >= 0
        Xtr, ytr = Xtr[keep], ytr[keep]
        probe = nn.Linear(X.shape[1], C).to(DEVICE)
        opt = torch.optim.Adam(probe.parameters(), lr=PROBE_LR)
        lossf = nn.CrossEntropyLoss()
        n = Xtr.shape[0]
        for ep in range(PROBE_EPOCHS):
            perm = torch.randperm(n)
            for b in range(0, n, PROBE_BATCH):
                idx = perm[b:b + PROBE_BATCH]
                xb = Xtr[idx].to(DEVICE); yb = ytr[idx].to(DEVICE)
                opt.zero_grad(); out = probe(xb); loss = lossf(out, yb); loss.backward(); opt.step()
        # ---- 4) eval on held-out sequences, per step ----
        with torch.no_grad():
            Xte = X[~is_train].to(DEVICE)
            pred = torch.argmax(probe(Xte), dim=1).cpu().numpy()
        y_te = y_all[~is_train]; step_te = steps[~is_train]
        correct = (pred == y_te) & (y_te >= 0)   # unseen-token labels count as incorrect
        dfL = pd.DataFrame({"step": step_te, "correct": correct.astype(int), "layer": L})
        agg = dfL.groupby("step", as_index=False).agg(accuracy=("correct", "mean"), n=("correct", "size"))
        agg["layer"] = L
        trained_rows.append(agg)
    trained_curve = pd.concat(trained_rows, ignore_index=True)
    trained_curve.to_csv(OUT_DIR / "trained_probe_masked_token_accuracy_curve.csv", index=False)
    print(trained_curve.groupby("layer")["accuracy"].mean())
else:
    print("RUN_TRAINED_PROBE is False; skipping trained-probe section.")

### Trained-probe plot (only if `RUN_TRAINED_PROBE = True`)

In [ ]:
if trained_curve is not None:
    with plt.style.context("default"):
        plt.figure(figsize=(9.2, 5.6))
        for i, L in enumerate(LAYERS_TO_PROBE):
            sub = trained_curve[trained_curve["layer"] == L].sort_values("step")
            tag = "Shallow" if L == min(LAYERS_TO_PROBE) else ("Deep" if L == max(LAYERS_TO_PROBE) else "Middle")
            plt.plot(sub["step"], sub["accuracy"], color=layer_color(L, i), linewidth=1.8,
                     label=f"Layer {L} ({tag})")
        plt.xlabel("Diffusion Timestep (t)")
        plt.ylabel("Masked Token Accuracy")
        plt.title("Final Token Prediction Probing Accuracy over Diffusion Timesteps (trained probe)")
        plt.ylim(0, 1.0)
        plt.grid(alpha=0.25, linestyle="--")
        plt.legend(loc="upper right")
        plt.tight_layout()
        plt.rcParams["pdf.fonttype"] = 42
        plt.savefig(OUT_DIR / "final_token_prediction_trained_probe.pdf", format="pdf", bbox_inches="tight")
        plt.show()
else:
    print("No trained-probe curve to plot.")

## 10. Findings summary

In [ ]:
mean_by_layer = ll_curve.groupby("layer")["accuracy"].mean().to_dict()
findings = {
    "config": {"model": MODEL_NAME, "num_layers": NUM_LAYERS, "layers_probed": LAYERS_TO_PROBE,
               "diffusion_steps": DIFFUSION_STEPS, "gen_len": GEN_LEN, "num_prompts": len(PROMPTS),
               "seed": SEED, "method": "logit_lens"},
    "logit_lens_mean_masked_accuracy_by_layer": {int(k): float(v) for k, v in mean_by_layer.items()},
    "expectation": "deep layers (high index) should exceed shallow layers for final-token prediction",
}
with open(OUT_DIR / "findings.json", "w") as f:
    json.dump(findings, f, indent=2)
print("==================== FINDINGS ====================")
print(json.dumps(findings, indent=2))
print("\nFiles in", OUT_DIR.resolve(), ":")
for p in sorted(OUT_DIR.iterdir()):
    print("  ", p.name)

In [ ]:
import shutil
shutil.make_archive("diffugpt_s_layer_token_probe_outputs", "zip", OUT_DIR)
try:
    from google.colab import files
    files.download("diffugpt_s_layer_token_probe_outputs.zip")
except Exception as exc:
    print("Download helper skipped:", exc)